# release16_failedfold_state_head_target_recipe_v1

## Governance

- Baselines: canonical runtime composer `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `release16_failedfold_state_feature_mining_v1`; blocked duplicate route `release16_guarded_reward_short_ppo_screen_v1`.
- Role: diagnose/target-recipe handoff, not promotion.
- Target windows: failed folds `wf_2020` and `wf_2023_2024`; guard fold `wf_2022` must receive zero new state-head target rows.
- Accepted trade-off: allow fold-local high-participation labels only when scope/lift/BC-readiness gates pass and no eval overlap exists.
- Changed layer: model-selection target generation / state representation.
- External reference: state-head auxiliary target contract based on transfer-stable `rulebase_reversal` feature group.
- Hypothesis: train-overlay labels can calibrate a `rulebase_reversal` state head that covers failed-fold missed-up-day states without using failed-test labels as training rows.
- Acceptance criteria: feature group is transfer-ready, target contract is fold-local, guard target rows are zero, lift semantics pass, BC readiness passes, and no actor is promoted.
- Expected failure mode: the rulebase state head marks guard-fold rows or creates no failed-fold target lift.
- Comparison-table row key: `release16_failedfold_state_head_target_recipe_v1`.

Closest success: `release16_failedfold_state_feature_mining_v1` selected `rulebase_reversal` as the best failed-test and transfer feature set.

Closest failure: `release16_guarded_reward_short_ppo_screen_v1` reused a drawdown-drift/trend reward patch and failed PPO-vs-BC drift, blocking longer PPO from that patch.

Why this is not a duplicate: this run does not change reward coefficients or rerun PPO; it converts the already-mined transfer-stable feature group into a fold-local target contract for the next checkpoint/BC screen.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
import sys

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "src" / "soxl_quant").exists():
        ROOT = candidate
        break
else:
    raise RuntimeError("could not locate repo root")
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.soxl_quant.app.state_objective_training_targets import (
    audit_state_objective_target_lift,
    audit_state_objective_target_scope,
    build_failedfold_state_head_target_contract,
    score_state_objective_bc_readiness,
    summarize_failedfold_state_head_target_contract,
    summarize_state_objective_bc_readiness,
    summarize_state_objective_target_lift_audit,
    summarize_state_objective_target_scope_audit,
)

EXPERIMENT = "release16_failedfold_state_head_target_recipe_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
SOURCE_CARD = ROOT / "runtime" / "model_selection" / "card" / "release16_failedfold_state_feature_mining_v1"
FAILED_FOLDS = ["wf_2020_train", "wf_2023_2024_train"]
GUARD_FOLDS = ["wf_2022_train"]
FEATURE_SET = "rulebase_reversal"
MIN_FEATURE_HITS = 5
MIN_TARGET_POSITION = 0.50
MAX_TARGET_POSITION = 0.65
OBJECTIVE_WEIGHT_BONUS = 4.0
CARD.mkdir(parents=True, exist_ok=True)
print(CARD)

/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_failedfold_state_head_target_recipe_v1


In [2]:
required_paths = {
    "source_summary": SOURCE_CARD / "summary.json",
    "source_selection": SOURCE_CARD / "selection.json",
    "feature_groups": SOURCE_CARD / "candidate_feature_groups.json",
    "train_overlay_frame": SOURCE_CARD / "train_overlay_state_frame.csv",
    "test_state_scores": SOURCE_CARD / "test_state_feature_set_scores.csv",
    "train_overlay_scores": SOURCE_CARD / "train_overlay_feature_set_scores.csv",
    "transfer_alignment": SOURCE_CARD / "state_transfer_feature_alignment.csv",
}
missing = [str(path.relative_to(ROOT)) for path in required_paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError(missing)

source_summary = json.loads(required_paths["source_summary"].read_text())
source_selection = json.loads(required_paths["source_selection"].read_text())
feature_groups = json.loads(required_paths["feature_groups"].read_text())
features = feature_groups[FEATURE_SET]
train_overlay = pd.read_csv(required_paths["train_overlay_frame"], parse_dates=["datetime"])
test_scores = pd.read_csv(required_paths["test_state_scores"])
train_scores = pd.read_csv(required_paths["train_overlay_scores"])
transfer_alignment = pd.read_csv(required_paths["transfer_alignment"])

preflight = {
    "source_decision": source_summary.get("decision"),
    "source_next_mechanism": source_summary.get("next_mechanism"),
    "source_best_test_feature_set": source_summary.get("best_test_feature_set"),
    "source_best_transfer_feature_set": source_summary.get("best_transfer_feature_set"),
    "source_state_feature_recipe_ready": bool(source_summary.get("state_feature_recipe_ready")),
    "features": features,
    "train_overlay_rows": int(len(train_overlay)),
}
print(json.dumps(preflight, indent=2, sort_keys=True))
if source_summary.get("best_transfer_feature_set") != FEATURE_SET:
    raise AssertionError("source did not select rulebase_reversal as best transfer feature set")
if not bool(source_summary.get("state_feature_recipe_ready")):
    raise AssertionError("source feature mining did not allow target recipe handoff")

{
  "features": [
    "RB_PANIC_SCORE_PROXY",
    "RB_DAILY_DROP_SEVERITY",
    "RB_CUMULATIVE_DROP_SEVERITY",
    "RB_RSI6_OVERSOLD",
    "RB_VOL_REGIME",
    "RB_BUY_COMPOSITE",
    "RB_SELL_COMPOSITE",
    "RB_PROFIT_TAKE_PROXY",
    "RB_VOLUME_STRESS",
    "RB_TREND_REVERSAL"
  ],
  "source_best_test_feature_set": "rulebase_reversal",
  "source_best_transfer_feature_set": "rulebase_reversal",
  "source_decision": "failedfold_state_head_features_ready_for_target_recipe_no_promotion",
  "source_next_mechanism": "build_rulebase_reversal_failedfold_state_head_target_recipe_then_checkpoint_screen",
  "source_state_feature_recipe_ready": true,
  "train_overlay_rows": 3731
}


In [3]:
score_join = test_scores.merge(
    train_scores,
    on="feature_set",
    suffixes=("_test", "_train_overlay"),
)
score_join = score_join[score_join["feature_set"] == FEATURE_SET].copy()
if score_join.empty:
    raise AssertionError("missing rulebase_reversal score row")
score_row = score_join.iloc[0].to_dict()
transfer_subset = transfer_alignment[transfer_alignment["feature_set"] == FEATURE_SET].copy()
transfer_subset = transfer_subset.sort_values("joint_abs_effect", ascending=False)
score_join.to_csv(CARD / "feature_set_gate.csv", index=False)
transfer_subset.to_csv(CARD / "rulebase_reversal_transfer_alignment.csv", index=False)
print(score_join[["feature_set", "holdout_auc_min_test", "holdout_accuracy_min_test", "holdout_auc_min_train_overlay", "holdout_accuracy_min_train_overlay"]].to_string(index=False))
print(transfer_subset.head(10)[["feature", "test_effect_size", "train_effect_size", "effect_sign_aligned", "joint_abs_effect"]].to_string(index=False))

      feature_set  holdout_auc_min_test  holdout_accuracy_min_test  holdout_auc_min_train_overlay  holdout_accuracy_min_train_overlay
rulebase_reversal              0.863622                   0.630952                       0.666731                            0.576004
                    feature  test_effect_size  train_effect_size  effect_sign_aligned  joint_abs_effect
          RB_SELL_COMPOSITE          0.489719           0.711590                 True          0.489719
           RB_RSI6_OVERSOLD         -0.400469          -0.359886                 True          0.359886
          RB_TREND_REVERSAL          0.357359           0.678399                 True          0.357359
RB_CUMULATIVE_DROP_SEVERITY          0.328998           0.713118                 True          0.328998
       RB_PROFIT_TAKE_PROXY          0.269991           0.781418                 True          0.269991
       RB_PANIC_SCORE_PROXY          0.156565          -0.146163                False          0.146163
    

In [4]:
contract = build_failedfold_state_head_target_contract(
    train_overlay,
    feature_columns=features,
    target_folds=FAILED_FOLDS,
    guard_folds=GUARD_FOLDS,
    min_feature_hits=MIN_FEATURE_HITS,
    min_target_position=MIN_TARGET_POSITION,
    max_target_position=MAX_TARGET_POSITION,
    objective_weight_bonus=OBJECTIVE_WEIGHT_BONUS,
)

fold_rows = []
for fold, group in train_overlay.groupby("fold", sort=True):
    first = group.iloc[0]
    fold_rows.append(
        {
            "name": fold,
            "train_start": first["train_start"],
            "train_end": first["train_end"],
            "eval_start": first["eval_start"],
            "eval_end": first["eval_end"],
        }
    )

scope_audit = audit_state_objective_target_scope(contract, folds=fold_rows)
scope_summary = summarize_state_objective_target_scope_audit(scope_audit)
lift_audit = audit_state_objective_target_lift(
    contract,
    baseline_risk_position_col="risk_target_position_original",
)
lift_summary = summarize_state_objective_target_lift_audit(lift_audit)
target_summary = summarize_failedfold_state_head_target_contract(
    contract,
    failed_folds=FAILED_FOLDS,
    guard_folds=GUARD_FOLDS,
)

contract.to_csv(CARD / "state_head_target_contract.csv", index=False)
scope_audit.to_csv(CARD / "target_scope_audit.csv", index=False)
lift_audit.to_csv(CARD / "target_lift_audit.csv", index=False)
print(json.dumps(target_summary, indent=2, sort_keys=True))
print(json.dumps(scope_summary, indent=2, sort_keys=True))
print(json.dumps(lift_summary, indent=2, sort_keys=True))

{
  "decision": "failedfold_state_head_target_ready_for_checkpoint_screen_no_promotion",
  "failed_target_rows": 1315,
  "folds": [
    "wf_2020_train",
    "wf_2022_train",
    "wf_2023_2024_train"
  ],
  "guard_target_rows": 0,
  "max_objective_weight": 25.0,
  "mean_target_lift": 0.26273921486481533,
  "next_mechanism": "run_state_head_checkpoint_or_bc_screen",
  "rows": 3731,
  "target_rows": 1315,
  "training_ready": true
}
{
  "audited_groups": 3,
  "blocked_groups": 0,
  "decision": "state_objective_targets_training_ready",
  "eval_overlap_rows": 0,
  "leakage_gate": true,
  "next_mechanism": "run_fold_local_state_objective_training_or_bc",
  "outside_train_eval_rows": 0,
  "scope_issues": [],
  "top_scope_issue": "",
  "training_allowed": true
}
{
  "audited_groups": 3,
  "blocked_groups": 0,
  "candidate_release_positive_lift_rows": 1369,
  "current_positive_rebound_lift_rows": 1368,
  "decision": "state_objective_target_lift_training_ready",
  "lift_semantics_gate": true,
  "

In [5]:
thresholds = json.loads(contract["failedfold_state_head_thresholds_json"].dropna().iloc[0])
threshold_rows = []
for feature, payload in thresholds.items():
    threshold_rows.append({"feature": feature, **payload})
threshold_table = pd.DataFrame(threshold_rows).sort_values("feature")
threshold_table.to_csv(CARD / "state_head_feature_thresholds.csv", index=False)

target_state = contract["failedfold_state_head_target_state"].fillna(False).astype(bool)
target_coverage = (
    contract.assign(target_state=target_state)
    .groupby("fold", dropna=False)
    .agg(
        rows=("datetime", "size"),
        target_rows=("target_state", "sum"),
        mean_hits=("failedfold_state_head_feature_hits", "mean"),
        mean_score=("failedfold_state_head_state_score", "mean"),
        mean_target_lift=("failedfold_state_head_target_lift", "mean"),
        max_objective_weight=("objective_weight", "max"),
    )
    .reset_index()
)
target_coverage["target_share"] = target_coverage["target_rows"] / target_coverage["rows"]
target_coverage.to_csv(CARD / "target_coverage_by_fold.csv", index=False)

if "action" in contract.columns:
    base_action = contract["action"].fillna("HOLD").astype(str)
else:
    base_action = pd.Series("HOLD", index=contract.index)
action_label = np.where(target_state, "STATE_HEAD_LIFT", base_action)
action_counts = pd.Series(action_label).value_counts(dropna=False).rename_axis("target_action").reset_index(name="rows")
action_counts["share"] = action_counts["rows"] / len(contract)
action_counts.to_csv(CARD / "target_action_distribution.csv", index=False)
shares = action_counts.set_index("target_action")["share"]
action_entropy = float(-(shares * np.log(shares)).sum())
bc_input = pd.DataFrame(
    {
        "profile": ["failedfold_state_head_rulebase_reversal"],
        "samples": [len(contract)],
        "unique_actions": [int(len(action_counts))],
        "dominant_action_share": [float(action_counts["share"].max())],
        "action_entropy": [action_entropy],
    }
)
bc_readiness = score_state_objective_bc_readiness(bc_input, target_scope="fold_local")
bc_summary = summarize_state_objective_bc_readiness(bc_readiness)
bc_readiness.to_csv(CARD / "bc_readiness_scores.csv", index=False)
print(target_coverage.to_string(index=False))
print(action_counts.to_string(index=False))
print(json.dumps(bc_summary, indent=2, sort_keys=True))

              fold  rows  target_rows  mean_hits  mean_score  mean_target_lift  max_objective_weight  target_share
     wf_2020_train  2470         1058   3.967206    0.396721          0.240302             25.000000      0.428340
     wf_2022_train   505            0   4.471287    0.447129          0.324139             25.000000      0.000000
wf_2023_2024_train   756          257   3.855820    0.385582          0.361837             19.648219      0.339947
  target_action  rows    share
           HOLD  2379 0.637631
STATE_HEAD_LIFT  1315 0.352452
            BUY    30 0.008041
           SELL     6 0.001608
    TAKE_PROFIT     1 0.000268
{
  "decision": "state_objective_bc_ready_for_training",
  "next_mechanism": "run_fold_local_bc_warm_start_then_walkforward_ppo",
  "selected_bc_readiness_gate": true,
  "selected_bc_readiness_loss": 0.0,
  "selected_dominant_action_share": 0.6376306620209059,
  "selected_fold_local_scope_gate": true,
  "selected_label_distribution_gate": true,
  "sele

In [6]:
training_ready = bool(
    target_summary["training_ready"]
    and scope_summary["training_allowed"]
    and lift_summary["training_allowed"]
    and bc_summary["selected_bc_readiness_gate"]
)
# This card is a target-recipe handoff only; actor retention requires a later official checkpoint/BC replay gate.
decision = (
    "release16_failedfold_state_head_target_recipe_ready_no_promotion"
    if training_ready
    else "release16_failedfold_state_head_target_recipe_blocked_no_training"
)
next_mechanism = (
    "run_rulebase_reversal_state_head_checkpoint_or_bc_screen_before_retention_training"
    if training_ready
    else "revise_failedfold_state_head_feature_group_or_target_scope"
)
selection = {
    "experiment": EXPERIMENT,
    "decision": decision,
    "next_mechanism": next_mechanism,
    "source_card": str(SOURCE_CARD.relative_to(ROOT)),
    "source_decision": source_summary.get("decision"),
    "feature_set": FEATURE_SET,
    "features": features,
    "failed_folds": FAILED_FOLDS,
    "guard_folds": GUARD_FOLDS,
    "min_feature_hits": MIN_FEATURE_HITS,
    "min_target_position": MIN_TARGET_POSITION,
    "max_target_position": MAX_TARGET_POSITION,
    "target_summary": target_summary,
    "scope_summary": scope_summary,
    "lift_summary": lift_summary,
    "bc_summary": bc_summary,
    "training_ready": training_ready,
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "ppo_allowed_next": False,
    "bc_or_checkpoint_screen_allowed_next": bool(training_ready),
}
summary = {
    **selection,
    "summary_points": [
        f"{FEATURE_SET} remains the transfer-stable state feature group from the source mining card.",
        f"State-head target rows: {target_summary['target_rows']} total, {target_summary['failed_target_rows']} failed-fold rows, {target_summary['guard_target_rows']} guard-fold rows.",
        f"Scope audit decision: {scope_summary['decision']} with eval_overlap_rows={scope_summary['eval_overlap_rows']}.",
        f"Lift audit decision: {lift_summary['decision']} with current_positive_rebound_lift_rows={lift_summary['current_positive_rebound_lift_rows']}.",
        "No actor is promoted or retained; this only permits a subsequent checkpoint/BC screen.",
    ],
    "sidecars": [
        "state_head_target_contract.csv",
        "target_scope_audit.csv",
        "target_lift_audit.csv",
        "target_coverage_by_fold.csv",
        "target_action_distribution.csv",
        "bc_readiness_scores.csv",
        "feature_set_gate.csv",
        "rulebase_reversal_transfer_alignment.csv",
        "state_head_feature_thresholds.csv",
        "selection.json",
        "summary.json",
    ],
}
(CARD / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n")
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n")
print(json.dumps(summary, indent=2, sort_keys=True))

{
  "actor_retention_allowed": false,
  "bc_or_checkpoint_screen_allowed_next": true,
  "bc_summary": {
    "decision": "state_objective_bc_ready_for_training",
    "next_mechanism": "run_fold_local_bc_warm_start_then_walkforward_ppo",
    "selected_bc_readiness_gate": true,
    "selected_bc_readiness_loss": 0.0,
    "selected_dominant_action_share": 0.6376306620209059,
    "selected_fold_local_scope_gate": true,
    "selected_label_distribution_gate": true,
    "selected_profile": "failedfold_state_head_rulebase_reversal",
    "selected_samples": 3731,
    "selected_target_scope": "fold_local",
    "selected_unique_actions": 5
  },
  "decision": "release16_failedfold_state_head_target_recipe_ready_no_promotion",
  "experiment": "release16_failedfold_state_head_target_recipe_v1",
  "failed_folds": [
    "wf_2020_train",
    "wf_2023_2024_train"
  ],
  "feature_set": "rulebase_reversal",
  "features": [
    "RB_PANIC_SCORE_PROXY",
    "RB_DAILY_DROP_SEVERITY",
    "RB_CUMULATIVE_DROP_SE

## Conclusion

Decision: `release16_failedfold_state_head_target_recipe_ready_no_promotion` if the executed `summary.json` records `training_ready=true`.

This card converts the prior `rulebase_reversal` state-feature mining result into a fold-local state-head target contract. It deliberately does not promote or retain an actor. The next permitted step is a checkpoint/BC screen that consumes `state_head_target_contract.csv`; PPO remains blocked until that screen rejects drift and passes official walk-forward gates.